# Archived historical analysis (NOT validated)
This notebook is preserved for provenance only. Its historical scores used row-level splits that can leak participant information, and its local absolute paths are not portable. Use the updated `nurse_stress` pipeline and grouped evaluation instead. Outputs have been removed to avoid redistributing participant-level excerpts.


**Academic origin:** University of Essex, CE888: Data Science and Decision Making (2022–2023 academic year). The maintained pipeline and documentation were refreshed in 2026.

Project:
*Nurse-stress-prediction-wearable-sensors*

Forecasting Stress Levels in Nurses by utilizing wearable
sensors

In [ ]:
import os
import shutil
import multiprocessing


#load main zip & unzip in datasetfile

In [ ]:
MAIN_PATH = "C:\\Users\\student\\Desktop\\stressdetection"

shutil.unpack_archive(MAIN_PATH + '\\Stress_dataset.zip', MAIN_PATH + '\\Data')
shutil.unpack_archive(MAIN_PATH + '\\Data\\Stress_dataset.zip', MAIN_PATH + '\\Data\\Stress_dataset')

stress_data_path = MAIN_PATH + '\\Data\\Stress_dataset'

cpu_count = int(multiprocessing.cpu_count()/2)
print(f'Using {cpu_count} CPUs')

new_list = [
    (file, sub_file) 
    for file in os.listdir(stress_data_path) 
    for sub_file in os.listdir(os.path.join(stress_data_path, file))
]

def unzip_parallel(file, sub_file):
    shutil.unpack_archive(
        os.path.join(stress_data_path, file, sub_file), 
        os.path.join(stress_data_path, file, sub_file[:-4])
    )

pool = multiprocessing.Pool(cpu_count)
results = pool.starmap(unzip_parallel, new_list)
pool.close()

#Combine all dataset


In [ ]:
# Script to combine all signals into one
import os
import pandas as pd

DATA_PATH = "C:\\Users\\student\\Desktop\\stressdetection\\Data"
SAVE_PATH = "C:\\Users\\student\\Desktop\\stressdetection\\processed_data1"

os.mkdir(SAVE_PATH)

final_columns = {
    'ACC': ['id', 'X', 'Y', 'Z', 'datetime'],
    'EDA': ['id', 'EDA', 'datetime'],
    'HR': ['id', 'HR', 'datetime'],
    'TEMP': ['id', 'TEMP', 'datetime'],
}

names = {
    'ACC.csv': ['X', 'Y', 'Z'],
    'EDA.csv': ['EDA'],
    'HR.csv': ['HR'],
    'TEMP.csv': ['TEMP'],
}

desired_signals = ['ACC.csv', 'EDA.csv', 'HR.csv', 'TEMP.csv']
desired_signals = ['ACC.csv', 'EDA.csv', 'HR.csv', 'TEMP.csv']

acc = pd.DataFrame(columns=final_columns['ACC'])
eda = pd.DataFrame(columns=final_columns['EDA'])
hr = pd.DataFrame(columns=final_columns['HR'])
temp = pd.DataFrame(columns=final_columns['TEMP'])

def process_df(df, file):
    start_timestamp = df.iloc[0,0]
    sample_rate = df.iloc[1,0]
    new_df = pd.DataFrame(df.iloc[2:].values, columns=df.columns)
    new_df['id'] =  file[-2:]
    new_df['datetime'] = [(start_timestamp + i/sample_rate) for i in range(len(new_df))]
    return new_df

for file in os.listdir(DATA_PATH):
    print(f'Processing {file}')
    for sub_file in os.listdir(os.path.join(DATA_PATH, file)):
        if not sub_file.endswith(".zip"):
            for signal in os.listdir(os.path.join(DATA_PATH, file, sub_file)):
                if signal in desired_signals:
                    df = pd.read_csv(os.path.join(DATA_PATH, file, sub_file, signal), names=names[signal], header=None)
                    if not df.empty:
                        if signal == 'ACC.csv':
                            acc = pd.concat([acc, process_df(df, file)])             
                        if signal == 'EDA.csv':
                            eda = pd.concat([eda, process_df(df, file)])
                        if signal == 'HR.csv':
                            hr = pd.concat([hr, process_df(df, file)])
                        if signal == 'TEMP.csv':
                            temp = pd.concat([temp, process_df(df, file)])

print('Saving Data ...')
acc.to_csv(os.path.join(SAVE_PATH, 'combined_acc.csv'), index=False)
eda.to_csv(os.path.join(SAVE_PATH, 'combined_eda.csv'), index=False)
hr.to_csv(os.path.join(SAVE_PATH, 'combined_hr.csv'), index=False)
temp.to_csv(os.path.join(SAVE_PATH, 'combined_temp.csv'), index=False)

#Merge All Data

In [ ]:
import pandas as pd
import os
import multiprocessing

COMBINED_DATA_PATH = "C:\\Users\\student\\Desktop\\stressdetection\\processed_data2"
SAVE_PATH = "C:\\Users\\student\\Desktop\\stressdetection\\merge"

if COMBINED_DATA_PATH != SAVE_PATH:
    os.mkdir(SAVE_PATH)

print("Reading data ...")

acc, eda, hr, temp = None, None, None, None

signals = ['acc', 'eda', 'hr', 'temp']

def read_parallel(signal):
    df = pd.read_csv(os.path.join(COMBINED_DATA_PATH, f"combined_{signal}.csv"), dtype={'id': str})
    return [signal, df]

pool = multiprocessing.Pool(len(signals))
results = pool.map(read_parallel, signals)
pool.close()
pool.join()

for i in results:
    globals()[i[0]] = i[1]

# Merge data
print('Merging Data ...')
ids = eda['id'].unique()
columns=['X', 'Y', 'Z', 'EDA', 'HR', 'TEMP', 'id', 'datetime']

def merge_parallel(id):
    print(f"Processing {id}")
    df = pd.DataFrame(columns=columns)
    
    acc_id = acc[acc['id'] == id]
    eda_id = eda[eda['id'] == id].drop(['id'], axis=1)
    hr_id = hr[hr['id'] == id].drop(['id'], axis=1)
    temp_id = temp[temp['id'] == id].drop(['id'], axis=1)

    df = acc_id.merge(eda_id, on='datetime', how='outer')
    df = df.merge(temp_id, on='datetime', how='outer')
    df = df.merge(hr_id, on='datetime', how='outer')

    df.fillna(method='ffill', inplace=True)
    df.fillna(method='bfill', inplace=True)

    return df

pool = multiprocessing.Pool(len(ids))
results = pool.map(merge_parallel, ids)
pool.close()
pool.join()

new_df = pd.concat(results, ignore_index=True)

print("Saving data ...")
new_df.to_csv(os.path.join(SAVE_PATH, "merged_data.csv"), index=False)

#compare each nurse csv with survey result file for the final result

In [ ]:
import pandas as pd
from datetime import timedelta, datetime
import numpy as np
import os
import multiprocessing

print("Reading 1 ...")
PATH = "C:\\Users\\student\\Desktop\\stressdetection\\merge1\\merged files"

df = pd.read_csv(os.path.join(PATH, 'merged_data_5C.csv'), dtype={'id': str})
df['datetime'] = pd.to_datetime(df['datetime'].apply(lambda x: x * (10 ** 9)))

print("Reading 2 ...")
survey_path = "C:\\Users\\student\\Desktop\\stressdetection\\SurveyResults.xlsx"

survey_df = pd.read_excel(survey_path, usecols=['ID', 'Start time', 'End time', 'date', 'Stress level'], dtype={'ID': str})
survey_df['Stress level'].replace('na', np.nan, inplace=True)
survey_df.dropna(inplace=True)
survey_df.replace({'missing_value': 'NaN'}, inplace=True)

survey_df['Start datetime'] = pd.to_datetime(survey_df['date'].map(str) + ' ' + survey_df['Start time'].map(str))
survey_df['End datetime'] = pd.to_datetime(survey_df['date'].map(str) + ' ' + survey_df['End time'].map(str))
survey_df.drop(['Start time', 'End time', 'date'], axis=1, inplace=True)

print("Converting ...")
daylight = pd.to_datetime(datetime(2020, 11, 1, 0, 0))

survey_df1 = survey_df[survey_df['End datetime'] <= daylight].copy()
survey_df1['Start datetime'] = survey_df1['Start datetime'].apply(lambda x: x + timedelta(hours=5))
survey_df1['End datetime'] = survey_df1['End datetime'].apply(lambda x: x + timedelta(hours=5))

survey_df2 = survey_df.loc[survey_df['End datetime'] > daylight].copy()
survey_df2['Start datetime'] = survey_df2['Start datetime'].apply(lambda x: x + timedelta(hours=6))
survey_df2['End datetime'] = survey_df2['End datetime'].apply(lambda x: x + timedelta(hours=6))

survey_df = pd.concat([survey_df1, survey_df2], ignore_index=True)

survey_df = survey_df.loc[survey_df['Stress level'] != 1.0]
survey_df.reset_index(drop=True, inplace=True)

print('Labelling ...')

if 'id' in df.columns:
    ids = df['id'].unique()
else:
    raise KeyError("'id' column not found in DataFrame 'df'.")

def parallel(id):
    new_df = pd.DataFrame(columns=['X', 'Y', 'Z', 'EDA', 'HR', 'TEMP', 'id', 'datetime', 'label'])

    sdf = df[df['id'] == id].copy()
    survey_sdf = survey_df[survey_df['ID'] == id].copy()

    for _, survey_row in survey_sdf.iterrows():
        ssdf = sdf[(sdf['datetime'] >= survey_row['Start datetime']) & (sdf['datetime'] <= survey_row['End datetime'])].copy()

        if not ssdf.empty:
            ssdf['label'] = np.repeat(survey_row['Stress level'], len(ssdf.index))
            new_df = pd.concat([new_df, ssdf], ignore_index=True)
        else:
            print(f"{survey_row['ID']} is missing label {survey_row['Stress level']} at {survey_row['Start datetime']} to {survey_row['End datetime']}")
        
    return new_df

if __name__ == '__main__':
    multiprocessing.freeze_support()

    pool = multiprocessing.Pool(len(ids))
    results = pool.map(parallel, ids)
    pool.close()
    pool.join()

    new_df = pd.concat(results, ignore_index=True)

    print('Saving ...')
    new_df.to_csv(os.path.join(PATH, 'new_merged_file_5c.csv'), index=False)
    print('Done')

#clean survey result file

In [ ]:
import pandas as pd

survey_path = "C:\\Users\\student\\Desktop\\stressdetection\\SurveyResults.xlsx"

# read survey results file and drop NaN values
survey_df = pd.read_excel(survey_path, usecols=['ID', 'Start time', 'End time', 'date', 'Stress level'], dtype={'ID': str})
survey_df['Stress level'].replace('na', np.nan, inplace=True)
survey_df.dropna(inplace=True)

# save updated dataframe to a new file
survey_df.to_csv("C:\\Users\\student\\Desktop\\stressdetection\\survey_results_cleaned.csv", index=False)
clean_df = pd.read_csv("C:\\Users\\student\\Desktop\\stressdetection\\survey_results_cleaned.csv")
clean_df

# Group the data by ID and get the row with the maximum temperature for each group
max_stres_rows = clean_df.loc[clean_df.groupby("ID")["Stress level"].idxmax()]

# Print the details of the row with the maximum temperature for each group
print(max_stres_rows)

#load all final labeled data in one csv

In [ ]:
import pandas as pd

path = "C:\\Users\\student\\Desktop\\stressdetection\\merge1\\Labeled Data"

# Get a list of all the CSV files in the directory
file_list = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv')]

# Load each CSV file into a separate DataFrame and store in a list
df_list = [pd.read_csv(f) for f in file_list]

# Concatenate the list of DataFrames into a single DataFrame
merged_df = pd.concat(df_list, ignore_index=True)
merged_df


In [ ]:
# Save the merged DataFrame as a single CSV file
merged_df.to_csv("merged_data.csv", index=False)

In [ ]:
merged_df.info()

In [ ]:
merged_df.nunique()

In [ ]:
# Group the data by ID and get the row with the maximum temperature for each group
max_temp_rows = merged_df.loc[merged_df.groupby("id")["TEMP"].idxmax()]

# Print the details of the row with the maximum temperature for each group
print(max_temp_rows)

In [ ]:
# Group the data by ID and calculate the maximum temperature for each group
max_temps = merged_df.groupby("id")["TEMP"].max()

# Convert the maximum temperature data to a pandas DataFrame
max_temps_merged_df = pd.DataFrame(max_temps)

# Create a boxplot with the maximum temperature data, with the ID on the x-axis
max_temps_merged_df.boxplot(column="TEMP", by="id")

# Set the axis labels and title
plt.xlabel("id")
plt.ylabel("Maximum Temperature")
plt.title("Maximum Temperature by ID")

# Display the plot
plt.show()

In [ ]:
# Group the data by ID and calculate the maximum hr for each group
max_hr = merged_df.groupby("id")["HR"].max()

# Convert the maximum edas data to a pandas DataFrame
max_hr_merged_df = pd.DataFrame(max_hr)

# Create a boxplot with the maximum hr data, with the ID on the x-axis
max_hr_merged_df.boxplot(column="HR", by="id")

# Set the axis labels and title
plt.xlabel("id")
plt.ylabel("Maximum HR")
plt.title("Maximum HR by ID")

# Display the plot
plt.show()

In [ ]:
# Group the data by ID and calculate the maximum eda for each group
max_eda = merged_df.groupby("id")["EDA"].max()

# Convert the maximum edas data to a pandas DataFrame
max_eda_merged_df = pd.DataFrame(max_eda)

# Create a boxplot with the maximum eda data, with the ID on the x-axis
max_eda_merged_df.boxplot(column="EDA", by="id")

# Set the axis labels and title
plt.xlabel("id")
plt.ylabel("Maximum EDA")
plt.title("Maximum EDA by ID")

# Display the plot
plt.show()

In [ ]:
import seaborn as sns

# Create a FacetGrid with one plot for each combination of ID and label
g = sns.FacetGrid(merged_df, col="id", hue="label", col_wrap=4, height=3.5, aspect=1.2)

# Map a histogram of the temperature data to each plot
g.map(sns.histplot, "TEMP", bins=20, kde=True, alpha=0.7, edgecolor='white')

# Set the axis labels and title
g.set_axis_labels("Temperature (°C)", "Frequency")
g.fig.suptitle("Temperature Distribution by ID and Label", y=1.05, fontsize=16, fontweight='bold')

# Add legend
g.add_legend(fontsize=12, title="Label", title_fontsize=14, frameon=True, edgecolor='black')

# Customize ticks and spines
g.despine(left=True, bottom=True)
g.set_xticklabels(fontsize=12)
g.set_yticklabels(fontsize=12)

# Display the plot
plt.show()


In [ ]:
import seaborn as sns

# Create a FacetGrid with one plot for each combination of ID and label
g = sns.FacetGrid(merged_df, col="id", hue="label", col_wrap=4, height=3.5, aspect=1.2)

# Map a histogram of the hr data to each plot
g.map(sns.histplot, "HR", bins=20, kde=True, alpha=0.7, edgecolor='white')

# Set the axis labels and title
g.set_axis_labels("HR", "Frequency")
g.fig.suptitle("HR Distribution by ID and Label", y=1.05, fontsize=16, fontweight='bold')

# Add legend
g.add_legend(fontsize=12, title="Label", title_fontsize=14, frameon=True, edgecolor='black')

# Customize ticks and spines
g.despine(left=True, bottom=True)
g.set_xticklabels(fontsize=12)
g.set_yticklabels(fontsize=12)

# Display the plot
plt.show()


In [ ]:
import seaborn as sns

# Create a FacetGrid with one plot for each combination of ID and label
g = sns.FacetGrid(merged_df, col="id", hue="label", col_wrap=4, height=3.5, aspect=1.2)

# Map a histogram of the EDA data to each plot
g.map(sns.histplot, "EDA", bins=20, kde=True, alpha=0.7, edgecolor='white')

# Set the axis labels and title
g.set_axis_labels("EDA", "Frequency")
g.fig.suptitle("EDA Distribution by ID and Label", y=1.05, fontsize=16, fontweight='bold')

# Add legend
g.add_legend(fontsize=12, title="Label", title_fontsize=14, frameon=True, edgecolor='black')

# Customize ticks and spines
g.despine(left=True, bottom=True)
g.set_xticklabels(fontsize=12)
g.set_yticklabels(fontsize=12)

# Display the plot
plt.show()


In [ ]:
import pandas as pd

# group by ID and calculate maximum temperature, HR, EDA, and stress level for each ID
temp_summary = merged_df.groupby("id")["TEMP"].max().describe()
hr_summary = merged_df.groupby("id")["HR"].max().describe()
eda_summary = merged_df.groupby("id")["EDA"].max().describe()
stress_summary = merged_df.groupby("id")["label"].max().describe()

# concatenate the data frames horizontally
summary_table = pd.concat([temp_summary, hr_summary, eda_summary, stress_summary], axis=1)

# rename the columns
summary_table.columns = ["Max Temp", "Max HR", "Max EDA", "Max Stress"]

# print the summary table
print("Summary statistics for each variable by ID:\n")
print(summary_table)


In [ ]:
import pandas as pd

# group by ID and calculate maximum temperature, HR, EDA, and stress level for each ID
max_temp = merged_df.groupby("id")["TEMP"].max().rename("Max Temp")
max_hr = merged_df.groupby("id")["HR"].max().rename("Max HR")
max_eda = merged_df.groupby("id")["EDA"].max().rename("Max EDA")
max_stress = merged_df.groupby("id")["label"].max().rename("Max Stress")

# concatenate the data frames horizontally
summary_table = pd.concat([max_temp, max_hr, max_eda, max_stress], axis=1)

# print the summary table for all IDs
print(summary_table)


#Scatter Plot

In [ ]:
import matplotlib.pyplot as plt

# Get the maximum values of temperature, HR, and EDA for each individual
max_temp_rows = merged_df.loc[merged_df.groupby("id")["TEMP"].idxmax()]
max_hr = merged_df.groupby("id")["HR"].max()
max_eda = merged_df.groupby("id")["EDA"].max()

# Create scatter plots for the relationships between maximum temperature, maximum HR, and maximum EDA with the stress level as the color
plt.scatter(max_temp_rows["TEMP"], max_hr, c=merged_df.groupby("id")["label"].max(), cmap="coolwarm")
plt.xlabel("Maximum Temperature")
plt.ylabel("Maximum HR")
plt.title("Maximum Temperature vs Maximum HR with Stress Level")
plt.colorbar()
plt.show()

plt.scatter(max_temp_rows["TEMP"], max_eda, c=merged_df.groupby("id")["label"].max(), cmap="coolwarm")
plt.xlabel("Maximum Temperature")
plt.ylabel("Maximum EDA")
plt.title("Maximum Temperature vs Maximum EDA with Stress Level")
plt.colorbar()
plt.show()

plt.scatter(max_hr, max_eda, c=merged_df.groupby("id")["label"].max(), cmap="coolwarm")
plt.xlabel("Maximum HR")
plt.ylabel("Maximum EDA")
plt.title("Maximum HR vs Maximum EDA with Stress Level")
plt.colorbar()
plt.show()


In [ ]:
# Group the data by ID and calculate the mean for each variable
mean_data = merged_df.groupby("id").mean()

# Create a bar plot for each variable, with the ID on the x-axis
mean_data.plot(kind="bar", y="TEMP", xlabel="ID", ylabel="Mean Temperature", title="Mean Temperature by ID")
mean_data.plot(kind="bar", y="HR", xlabel="ID", ylabel="Mean HR", title="Mean HR by ID")
mean_data.plot(kind="bar", y="EDA", xlabel="ID", ylabel="Mean EDA", title="Mean EDA by ID")
mean_data.plot(kind="bar", y="label", xlabel="ID", ylabel="Mean Stress Level", title="Mean Stress Level by ID")

# Display the plots
plt.show()


In [ ]:
# Calculate the average value of each variable for each ID
avg_df = merged_df.groupby("id").mean()
print (avg_df)

In [ ]:
# Count the number of occurrences of each label for each ID
label_counts = merged_df.groupby(['id', 'label']).size().unstack()

# Create a stacked bar chart
label_counts.plot(kind='bar', stacked=True)

# Set the axis labels and title
plt.xlabel('id')
plt.ylabel('Count')
plt.title('Stress Level by ID')

# Display the plot
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a figure with 3 subplots arranged horizontally
fig, axs = plt.subplots(1, 3, figsize=(15,5))

# Create a box plot of temperature for each ID, colored by label
sns.boxplot(x="id", y="EDA", hue="label", data=merged_df, ax=axs[0])
sns.boxplot(x="id", y="HR", hue="label", data=merged_df, ax=axs[1])
sns.boxplot(x="id", y="TEMP", hue="label", data=merged_df, ax=axs[2])

# Set the axis labels and title for each subplot
axs[0].set_xlabel("ID")
axs[0].set_ylabel("EDA")
axs[0].set_title("EDA Distribution by ID and Label")

axs[1].set_xlabel("ID")
axs[1].set_ylabel("HR")
axs[1].set_title("HR Distribution by ID and Label")

axs[2].set_xlabel("ID")
axs[2].set_ylabel("Temperature")
axs[2].set_title("Temperature Distribution by ID and Label")

# Adjust the layout of the subplots to avoid overlapping
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
# Select the columns of interest
cols_of_interest = ["TEMP", "HR", "EDA", "label"]

# Create a pairplot with the selected columns
sns.pairplot(data=merged_df[cols_of_interest])

In [ ]:
# Calculate the correlation coefficients between the variables TEMP, HR, EDA with id and label
corr_matrix = merged_df[["id", "label", "TEMP", "HR", "EDA"]].corr()

# Plot the correlation matrix as a heatmap
import seaborn as sns
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')

# Display the plot
plt.show()

print(corr_matrix)


In [ ]:
if abs(corr_temp) >= 0.2:
    print("There is a moderate correlation between temperature and stress level.")
elif abs(corr_temp) >= 0.1:
    print("There is a weak correlation between temperature and stress level.")
else:
    print("There is no correlation between temperature and stress level.")
    
if abs(corr_hr) >= 0.2:
    print("There is a moderate correlation between HR and stress level.")
elif abs(corr_hr) >= 0.1:
    print("There is a weak correlation between HR and stress level.")
else:
    print("There is no correlation between HR and stress level.")
    
if abs(corr_eda) >= 0.2:
    print("There is a moderate correlation between EDA and stress level.")
elif abs(corr_eda) >= 0.1:
    print("There is a weak correlation between EDA and stress level.")
else:
    print("There is no correlation between EDA and stress level.")

In [ ]:
import pandas as pd

# Group data by stress level and calculate summary statistics for maximum temperature, HR, and EDA
temp_summary = merged_df.groupby("label")["TEMP"].max().describe()
hr_summary = merged_df.groupby("label")["HR"].max().describe()
eda_summary = merged_df.groupby("label")["EDA"].max().describe()

# Create a dataframe with the summary statistics
summary_df = pd.DataFrame({
    "Low Stress": [temp_summary["min"], hr_summary["min"], eda_summary["min"]],
    "Medium Stress": [temp_summary["50%"], hr_summary["50%"], eda_summary["50%"]],
    "High Stress": [temp_summary["max"], hr_summary["max"], eda_summary["max"]]
}, index=["Max Temperature", "Max HR", "Max EDA"])

# Print the summary table
print(summary_df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Group data by stress level and calculate summary statistics for maximum temperature, HR, and EDA
temp_summary = merged_df.groupby("label")["TEMP"].max().describe()
hr_summary = merged_df.groupby("label")["HR"].max().describe()
eda_summary = merged_df.groupby("label")["EDA"].max().describe()

# Create a dataframe with the summary statistics
summary_df = pd.DataFrame({
    "Low Stress": [temp_summary["min"], hr_summary["min"], eda_summary["min"]],
    "Medium Stress": [temp_summary["50%"], hr_summary["50%"], eda_summary["50%"]],
    "High Stress": [temp_summary["max"], hr_summary["max"], eda_summary["max"]]
}, index=["Max Temperature", "Max HR", "Max EDA"])

# Plot the summary table as a bar plot
ax = summary_df.plot(kind="bar", figsize=(8, 6), rot=0)

# Set plot attributes
ax.set_xlabel("Physiological Measures")
ax.set_ylabel("Max Values")
ax.set_title("Summary Statistics by Stress Level")
ax.legend(title="Stress Level")

# Save plot as an image
plt.savefig("summary_table.png", dpi=300, bbox_inches="tight")


*Machine Learning Model*

**Random Forest Classifier**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, mean_squared_error

X = merged_df[['X', 'Y', 'Z', 'TEMP', 'HR', 'EDA']] # input variables
y = merged_df['label'] # target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a random forest classifier on the training data
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate the model on the testing data
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

print('Precision:', precision_score(y_test, y_pred, average='weighted'))
print('Recall:', recall_score(y_test, y_pred, average='weighted'))
print('F1-score:', f1_score(y_test, y_pred, average='weighted'))

# Evaluate the model performance using R-squared and RMSE metrics
print('R-squared score for id prediction:', r2_score(y_test, y_pred))
print('RMSE for id prediction:', mean_squared_error(y_test, y_pred, squared=False))



In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import seaborn as sns

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

print('Confusion Matrix:', confusion_matrix(y_test, y_pred))
print('Classification Report:', classification_report(y_test, y_pred))
print('ROC AUC Score:', roc_auc_score(y_test, model.predict(X_test), multi_class='ovr', average='weighted'))

# Create a heatmap of the confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()


**Decision Tree Classifier**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, mean_squared_error

X = merged_df[['X', 'Y', 'Z', 'TEMP', 'HR', 'EDA']] # input variables
y = merged_df['label'] # target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.80, random_state=42)


clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: {:.2f}%".format(accuracy*100))

# Calculate precision
prec = precision_score(y_test, y_pred, average='weighted')

# Calculate recall
rec = recall_score(y_test, y_pred, average='weighted')

# Calculate F1 score
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1 Score: {f1:.3f}")

# Evaluate the model performance using R-squared and RMSE metrics
print('R-squared score for id prediction:', r2_score(y_test, y_pred))
print('RMSE for id prediction:', mean_squared_error(y_test, y_pred, squared=False))

importances = clf.feature_importances_
for feature, importance in zip(X.columns, importances):
    print(f"{feature}: {importance}")


In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(10, 8))
plot_tree(clf, filled=True, feature_names=['TEMP', 'HR', 'EDA'], class_names=['low', 'medium', 'high'])
plt.show()


**Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Split the data into training and testing sets
X = merged_df[['X', 'Y', 'Z', 'TEMP', 'HR', 'EDA']] # input variables
y = merged_df['label'] # target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the data using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the logistic regression model with increased max_iter and different solver
clf = LogisticRegression(random_state=42, max_iter=1000, solver='saga')
clf.fit(X_train_scaled, y_train)

# Predict the labels for the test set
y_pred = clf.predict(X_test_scaled)

# Evaluate the model performance using accuracy score, precision score, recall score, and F1-score
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=1)
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Evaluate the model performance using R-squared and RMSE metrics
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

# Print the results
print("Accuracy:", accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)
print('R-squared score for id prediction:', r2)
print('RMSE for id prediction:', rmse)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix: ")
print(cm)

import seaborn as sns

# Create a heatmap of the confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

# Generate classification report
cr = classification_report(y_test, y_pred, zero_division=1)
print("Classification Report: ")
print(cr)

**k-Nearest Neighbors (kNN)**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, mean_squared_error

X = merged_df[['X', 'Y', 'Z', 'TEMP', 'HR', 'EDA']] # input variables
y = merged_df['label'] # target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred, average='weighted'))
print('Recall:', recall_score(y_test, y_pred, average='weighted'))
print('F1-score:', f1_score(y_test, y_pred, average='weighted'))

# Evaluate the model performance using R-squared and RMSE metrics
print('R-squared score for id prediction:', r2_score(y_test, y_pred))
print('RMSE for id prediction:', mean_squared_error(y_test, y_pred, squared=False))


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(classification_report(y_test, y_pred))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

**Naive Bayes Classifier**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, mean_squared_error

X = merged_df[['X', 'Y', 'Z', 'TEMP', 'HR', 'EDA']] # input variables
y = merged_df['label'] # target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42)

nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred = nb.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred, average='weighted'))
print('Recall:', recall_score(y_test, y_pred, average='weighted'))
print('F1-score:', f1_score(y_test, y_pred, average='weighted'))

# Evaluate the model performance using R-squared and RMSE metrics
print('R-squared score for id prediction:', r2_score(y_test, y_pred))
print('RMSE for id prediction:', mean_squared_error(y_test, y_pred, squared=False))


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(classification_report(y_test, y_pred))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

**Neural Networks**

In [ ]:
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, mean_squared_error
import numpy as np

# Split data into training and testing sets
X = merged_df[['X', 'Y', 'Z', 'TEMP', 'HR', 'EDA']].values # input variables
y = merged_df['label'].values # target variable
y = to_categorical(y) # one-hot encoding of target variable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build and train the neural network
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=6))
model.add(Dense(64, activation='relu'))
model.add(Dense(3, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

# Evaluate the model on the test set
y_pred = np.argmax(model.predict(X_test), axis=-1)
y_test = np.argmax(y_test, axis=-1)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred, average='weighted'))
print('Recall:', recall_score(y_test, y_pred, average='weighted'))
print('F1-score:', f1_score(y_test, y_pred, average='weighted'))

# Evaluate the model performance using R-squared and RMSE metrics
print('R-squared score for id prediction:', r2_score(y_test, y_pred))
print('RMSE for id prediction:', mean_squared_error(y_test, y_pred, squared=False))


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(classification_report(y_test, y_pred))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

Result:

Six machine learning models were trained on a dataset containing six physiological variables to predict different levels of stress. The Random Forest Classifier achieved high accuracy (99.64%) and good performance on precision, recall, and F1-score. The Logistic Regression model showed poor performance with a negative R-squared score. The KNN Classifier model outperformed the logistic regression model. The Neural Network achieved an accuracy of 0.84 on the test set.

EDA was found to be the most important variable, followed by HR, Z, and X, in detecting stress. The study recommends that the company should address the class imbalance in the dataset, assess the relevance of physiological signals in detecting stress accurately, optimize for recall, and ensure that the algorithm's decision-making process is transparent and explainable to end-users.